# Quantization Backdoor 


In [ ]:
from pathlib import Path
import csv
import json
import os
import re
import shutil
import subprocess
import sys
import time
import zipfile

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")


def find_repo_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "defense").exists() and (candidate / "record").exists() and (candidate / "notebooks_finetuning").exists():
            return candidate
    raise FileNotFoundError("Could not locate the cloned backdoor_finetuning repo. Start Jupyter from the repo or its notebooks_finetuning folder.")


REPO = find_repo_root()
NOTEBOOK_DIR = REPO / "notebooks_finetuning"
QUANTIZATION_DIR = NOTEBOOK_DIR / "quantization"
MODEL_ZIP_SEARCH_DIRS = [QUANTIZATION_DIR, NOTEBOOK_DIR]
ZIP_SEARCH_DIRS = [QUANTIZATION_DIR, NOTEBOOK_DIR]

PYTHON_EXE = sys.executable


def unique_zips_by_name(search_dirs):
    zips = {}
    for folder in search_dirs:
        if not folder.exists():
            continue
        for path in sorted(folder.glob("*.zip")):
            zips.setdefault(path.name, path)
    return zips


AVAILABLE_ZIP_PATHS = unique_zips_by_name(ZIP_SEARCH_DIRS)
AVAILABLE_ZIPS = sorted(AVAILABLE_ZIP_PATHS)
AVAILABLE_MODEL_ZIP_PATHS = {
    name: path
    for name, path in unique_zips_by_name(MODEL_ZIP_SEARCH_DIRS).items()
    if name.startswith("cifar10_")
}
AVAILABLE_MODEL_ZIPS = sorted(AVAILABLE_MODEL_ZIP_PATHS)

RUN_ZIPS = "ALL_MODEL_ZIPS"

MODEL_ZIP_NAME = None

RESULT_FILE = None

def pick_device():
    if torch.cuda.is_available():
        return "cuda"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


DEVICE = pick_device()
BATCH_SIZE = 256
NUM_WORKERS = 0
THRESHOLD = 0.5

MAX_SAMPLES = None

REUSE_EXISTING_RESULTS = True


SAVE_QUANTIZED_MODELS = False

QUANTIZATION_BITS = [8, 6, 4, 3, 2]


def resolve_zip_path(zip_choice):
    zip_path = Path(zip_choice).expanduser()
    if zip_path.is_absolute():
        return zip_path
    for folder in ZIP_SEARCH_DIRS:
        candidate = folder / zip_path
        if candidate.exists():
            return candidate
    return QUANTIZATION_DIR / zip_path


def resolve_zip_paths(zip_choices):
    if zip_choices == "ALL_MODEL_ZIPS":
        return [AVAILABLE_MODEL_ZIP_PATHS[name] for name in AVAILABLE_MODEL_ZIPS]
    if isinstance(zip_choices, (str, Path)):
        return [resolve_zip_path(zip_choices)]
    return [resolve_zip_path(choice) for choice in zip_choices]


SOURCE_ZIPS = resolve_zip_paths(RUN_ZIPS)
OUTPUT_DIR = QUANTIZATION_DIR / "outputs"
CACHE_DIR = REPO / "record" / "_model_zip_cache"
COMBINED_CSV_OUT = OUTPUT_DIR / "quantization_results_selected_models.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REPO, PYTHON_EXE, len(SOURCE_ZIPS), MODEL_ZIP_NAME, DEVICE, OUTPUT_DIR, AVAILABLE_MODEL_ZIPS


## 1. Zip Check

In [ ]:
assert REPO.exists(), f"Repo folder missing: {REPO}"
assert Path(PYTHON_EXE).exists(), f"Python executable missing: {PYTHON_EXE}"

print("Available model zips for overnight mode:")
for name in AVAILABLE_MODEL_ZIPS:
    print(" -", name)

if RESULT_FILE:
    print("Using existing record folder:", REPO / "record" / RESULT_FILE)
else:
    assert SOURCE_ZIPS, "No source zips selected. Put cifar10_*.zip files in notebooks_finetuning/quantization, or set RUN_ZIPS to a filename/list."
    print(f"Selected {len(SOURCE_ZIPS)} zip(s):")
    for source_zip in SOURCE_ZIPS:
        assert source_zip.exists(), f"Zip missing: {source_zip}. Pick one of AVAILABLE_ZIPS in the first cell."
        print(" -", source_zip)
        with zipfile.ZipFile(source_zip) as zf:
            inner_zips = [info.filename for info in zf.infolist() if info.filename.lower().endswith(".zip")]
        if inner_zips:
            print("   Inner model zips:")
            for name in inner_zips:
                print("    -", name)


## 2. Prepare The Selected Model(s)



In [ ]:
def _zip_entries(zip_path):
    with zipfile.ZipFile(zip_path) as zf:
        return [info for info in zf.infolist() if not info.is_dir()]


def _inner_model_zip_names(bundle_zip):
    return [info.filename for info in _zip_entries(bundle_zip) if info.filename.lower().endswith(".zip")]


def _materialize_inner_zip(bundle_zip, inner_name):
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    out_path = CACHE_DIR / Path(inner_name).name
    with zipfile.ZipFile(bundle_zip) as zf:
        info = zf.getinfo(inner_name)
        if out_path.exists() and out_path.stat().st_size == info.file_size:
            return out_path
        print(f"Extracting inner zip {inner_name} -> {out_path}")
        with zf.open(info) as src, open(out_path, "wb") as dst:
            shutil.copyfileobj(src, dst)
    return out_path


def selected_model_archives_for_source(source_zip):
    inner_names = _inner_model_zip_names(source_zip)
    if not inner_names:
        return [source_zip]
    if MODEL_ZIP_NAME == "ALL":
        return [_materialize_inner_zip(source_zip, name) for name in inner_names]
    if MODEL_ZIP_NAME is None:
        raise ValueError(f"{source_zip} contains multiple model zips. Set MODEL_ZIP_NAME to one name, or 'ALL'.")
    matches = [name for name in inner_names if Path(name).name == MODEL_ZIP_NAME or name == MODEL_ZIP_NAME]
    if not matches:
        raise ValueError(f"MODEL_ZIP_NAME={MODEL_ZIP_NAME!r} was not found in {source_zip}. Run the listing cell above and copy one exact name.")
    return [_materialize_inner_zip(source_zip, matches[0])]


def selected_model_archives():
    if RESULT_FILE:
        return []
    archives = []
    for source_zip in SOURCE_ZIPS:
        archives.extend(selected_model_archives_for_source(source_zip))
    return archives


def infer_result_file(model_zip):
    return Path(model_zip).stem


def normalize_extracted_record(record_dir):
    attack_result = record_dir / "attack_result.pt"
    if attack_result.exists():
        return
    matches = list(record_dir.rglob("attack_result.pt"))
    if len(matches) != 1:
        raise FileNotFoundError(f"Could not find one attack_result.pt under {record_dir}; found {len(matches)}")
    nested = matches[0].parent
    print(f"Normalizing nested extracted folder {nested} -> {record_dir}")
    for child in nested.iterdir():
        target = record_dir / child.name
        if target.exists():
            raise FileExistsError(f"Cannot move {child}; target already exists: {target}")
        shutil.move(str(child), str(target))


def prepare_record_from_zip(model_zip):
    result_file = infer_result_file(model_zip)
    record_dir = REPO / "record" / result_file
    attack_result = record_dir / "attack_result.pt"
    if not attack_result.exists():
        record_dir.mkdir(parents=True, exist_ok=True)
        print(f"Extracting {Path(model_zip).name} -> {record_dir}")
        with zipfile.ZipFile(model_zip) as zf:
            zf.extractall(record_dir)
        normalize_extracted_record(record_dir)
    assert attack_result.exists(), f"Attack result missing after extraction: {attack_result}"
    assert (record_dir / "bd_test_dataset").exists(), f"bd_test_dataset missing: {record_dir}"
    return result_file, record_dir


def prepare_selected_records():
    if RESULT_FILE:
        record_dir = REPO / "record" / RESULT_FILE
        assert (record_dir / "attack_result.pt").exists(), f"Attack result missing: {record_dir / 'attack_result.pt'}"
        assert (record_dir / "bd_test_dataset").exists(), f"bd_test_dataset missing: {record_dir}"
        return [(RESULT_FILE, record_dir)]
    return [prepare_record_from_zip(path) for path in selected_model_archives()]


selected_records = prepare_selected_records()
print("Selected records:")
for result_file, record_dir in selected_records:
    print(" -", result_file, "=>", record_dir)


## 3. Command Helpers

In [ ]:
def _extract_json(stdout):
    text = stdout.strip()
    start = text.find('{')
    end = text.rfind('}')
    if start == -1 or end == -1 or end <= start:
        raise ValueError("No JSON object found in command output:\n" + stdout[-2000:])
    return json.loads(text[start:end + 1])


def run_cmd(args):
    started = time.time()
    env = os.environ.copy()
    env.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
    proc = subprocess.run(
        [PYTHON_EXE, *args],
        cwd=str(REPO),
        text=True,
        capture_output=True,
        env=env,
    )
    elapsed = time.time() - started
    if proc.returncode != 0:
        print(proc.stdout)
        print(proc.stderr)
        raise RuntimeError(f"Command failed with return code {proc.returncode}: {' '.join(args)}")
    result = _extract_json(proc.stdout)
    result["elapsed_seconds"] = elapsed
    return result


def sample_args():
    args = []
    if MAX_SAMPLES is not None:
        args += ["--max_samples", str(MAX_SAMPLES)]
    return args


def output_dir_for(result_file):
    out_dir = OUTPUT_DIR / result_file
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir


def bits_label(bits):
    return f"{bits}bit"


def quantization_json_path(result_file, bits):
    return output_dir_for(result_file) / f"quantization_weight_quantize_{bits_label(bits)}.json"


def reusable_quantization_result(result_file, bits):
    if not REUSE_EXISTING_RESULTS:
        return None
    path = quantization_json_path(result_file, bits)
    if not path.exists():
        return None
    with open(path) as f:
        metrics = json.load(f)
    if metrics.get("max_samples") != MAX_SAMPLES:
        return None
    metrics["elapsed_seconds"] = 0.0
    print(f"Reusing existing quantization metrics: {path}")
    return metrics



def check_baseline(result_file):
    return run_cmd([
        "defense/check_backdoor.py",
        "--result_file", result_file,
        "--device", DEVICE,
        "--batch_size", str(BATCH_SIZE),
        "--num_workers", str(NUM_WORKERS),
        "--threshold", str(THRESHOLD),
        "--json",
        *sample_args(),
    ])


def run_quantization(result_file, bits):
    cached = reusable_quantization_result(result_file, bits)
    if cached is not None:
        return cached
    args = [
        "defense/quantize_and_check.py",
        "--result_file", result_file,
        "--mode", "weight_quantize",
        "--bits", str(bits),
        "--output_dir", str(output_dir_for(result_file)),
        "--device", DEVICE,
        "--batch_size", str(BATCH_SIZE),
        "--num_workers", str(NUM_WORKERS),
        "--threshold", str(THRESHOLD),
        "--json",
        *sample_args(),
    ]
    if not SAVE_QUANTIZED_MODELS:
        args.append("--no_save_model")
    return run_cmd(args)


## 4. Run Quantization Suite


In [ ]:
def csv_name_for(result_file):
    match = re.match(r"(.+)_([0-9]+(?:_[0-9]+)?)$", result_file)
    if match:
        model_part, poison_part = match.groups()
        return f"quantization_results_{model_part}_poison_{poison_part}.csv"
    return f"quantization_results_{result_file}.csv"


def write_rows(csv_path, rows):
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        return
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


def run_suite_for_record(result_file, record_dir):
    print("=" * 88)
    print("Running", result_file)
    print("=" * 88)
    baseline = check_baseline(result_file)
    rows = [{
        "result_file": result_file,
        "quantization_mode": "baseline",
        "bits": None,
        "max_samples": MAX_SAMPLES,
        "clean_acc_before": baseline["clean_acc"],
        "asr_before": baseline["asr"],
        "clean_acc_after": baseline["clean_acc"],
        "asr_after": baseline["asr"],
        "clean_acc_delta": 0.0,
        "asr_delta": 0.0,
        "persistence_ratio": 1.0,
        "backdoor_present_after": baseline["backdoor_present"],
        "quantized_parameter_fraction": 0.0,
        "mean_abs_weight_error": 0.0,
        "estimated_weight_size_ratio": 1.0,
        "elapsed_seconds": baseline["elapsed_seconds"],
        "metrics_source": "baseline checker",
    }]
    print("Baseline clean/asr:", baseline["clean_acc"], baseline["asr"])

    for bits in QUANTIZATION_BITS:
        print(f"Running {result_file}: weight_quantize bits={bits} ...")
        metrics = run_quantization(result_file, bits)
        row = {
            "result_file": result_file,
            "quantization_mode": metrics["mode"],
            "bits": metrics["bits"],
            "max_samples": metrics["max_samples"],
            "clean_acc_before": metrics["before"]["clean_acc"],
            "asr_before": metrics["before"]["asr"],
            "clean_acc_after": metrics["after"]["clean_acc"],
            "asr_after": metrics["after"]["asr"],
            "clean_acc_delta": metrics["delta"]["clean_acc"],
            "asr_delta": metrics["delta"]["asr"],
            "persistence_ratio": metrics["persistence_ratio"],
            "backdoor_present_after": metrics["after"]["backdoor_present"],
            "quantized_parameter_fraction": metrics["quantization"]["quantized_parameter_fraction"],
            "mean_abs_weight_error": metrics["quantization"]["mean_abs_weight_error"],
            "estimated_weight_size_ratio": metrics["quantization"]["estimated_weight_size_ratio"],
            "elapsed_seconds": metrics["elapsed_seconds"],
            "metrics_source": metrics["save_path"],
        }
        rows.append(row)
        print("  clean/asr after:", row["clean_acc_after"], row["asr_after"], "present:", row["backdoor_present_after"])

    csv_out = output_dir_for(result_file) / csv_name_for(result_file)
    write_rows(csv_out, rows)
    print("Saved model CSV:", csv_out)
    return rows, csv_out


all_rows = []
csv_paths = []
for result_file, record_dir in selected_records:
    rows, csv_out = run_suite_for_record(result_file, record_dir)
    all_rows.extend(rows)
    csv_paths.append(csv_out)

if len(selected_records) > 1:
    write_rows(COMBINED_CSV_OUT, all_rows)
    print("Saved combined CSV:", COMBINED_CSV_OUT)

print("Output folder:", OUTPUT_DIR)
print("Done.")


## 5. View Results

In [ ]:
try:
    import pandas as pd
    df = pd.DataFrame(all_rows)
    compact = df[["result_file", "quantization_mode", "bits", "clean_acc_after", "asr_after", "backdoor_present_after"]]
    try:
        display(compact)
    except NameError:
        print(compact)
except ImportError:
    for row in all_rows:
        print(row["result_file"], row["quantization_mode"], row["bits"], row["clean_acc_after"], row["asr_after"], row["backdoor_present_after"])

print("CSV files:")
for path in csv_paths:
    print(" -", path)
if len(selected_records) > 1:
    print(" -", COMBINED_CSV_OUT)
